# Data Exploration

## Purpose
Transform the raw dataset into a cleaned, feature-engineered dataset aligned for training and future Vertex AI deployment.

## Imports & Configuration

In [42]:
# Standard libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

## Load Project Secrets (Colab / userdata)

In [43]:
from google.colab import userdata

PROJECT_ID = userdata.get("GCP_PROJECT_ID")
GITHUB_USER = userdata.get("GITHUB_USER")
GCS_BUCKET = userdata.get("GCS_BUCKET")
TRAINING_PREFIX = userdata.get("TRAINING_PREFIX")
DATA_PREP_PREFIX = userdata.get("DATA_PREP_PREFIX")
REPO_NAME = userdata.get("REPO_NAME")
REGION = userdata.get("REGION")

REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
CLONE_PATH = f"/content/{REPO_NAME}"

## Authenticate & Configure gcloud

In [44]:
from google.colab import auth
auth.authenticate_user()  # Prompts OAuth login

!gcloud auth login --quiet
!gcloud config set project $PROJECT_ID
!gcloud config set compute/region $REGION

print("Project:")
!gcloud config get-value project
print("\nAuthenticated User Account:")
!gcloud config get-value account
print("\nRegion:")
!gcloud config get-value compute/region

Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=32555940559.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fappengine.admin+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcompute+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Faccounts.reauth&state=6viyr3BgbhHTOROFA9XqEbWWFHqnB8&prompt=consent&token_usage=remote&access_type=offline&code_challenge=t3AfEefFLqX9yen6-aKaecwF0Vs6QLH7tq5lEZLglAs&code_challenge_method=S256

Once finished, enter the verification code provided in your browser: 4/0AfrIepA4ZpmPTMdnJMf-5_9PrwHxOR5n-3vXk1CLm0Bq-6GQX6-uicin52vw7I-_gID7FQ

You are now logged in as [davistyrant@gmail.com].
Your current project 

## Clone Repository

In [45]:
%cd /content
!rm -rf {REPO_NAME}
!git clone {REPO_URL}

# Move to repo root
%cd {CLONE_PATH}

# Verify structure
!ls -lh data/raw

/content
Cloning into 'gcp-ml-engineer-vertex-ai'...
remote: Enumerating objects: 150, done.
remote: Counting objects: 100% (150/150), done.
remote: Compressing objects: 100% (87/87), done.
remote: Total 150 (delta 66), reused 129 (delta 50), pack-reused 0 (from 0)
Receiving objects: 100% (150/150), 62.77 KiB | 10.46 MiB/s, done.
Resolving deltas: 100% (66/66), done.
/content/gcp-ml-engineer-vertex-ai
total 60K
-rw-r--r-- 1 root root 59K Feb 26 18:35 train.csv


## Load Titanic CSV

In [46]:
# Path to Titanic CSV relative to repo root
DATA_PATH = "data/raw/train.csv"

# Load CSV
df = pd.read_csv(DATA_PATH)

# Quick preview
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## Dataset Overview

In [47]:
print("Shape:", df.shape)
print("\nData Types:")
print(df.dtypes)
df.describe(include="all")

Shape: (891, 12)

Data Types:
PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
count,891.000000,891.000000,891.000000,891,891,714.000000,891.000000,891.000000,891,891.000000,204,889
unique,NaN,NaN,NaN,891,2,NaN,NaN,NaN,681,NaN,147,3
top,NaN,NaN,NaN,"Dooley, Mr. Patrick",male,NaN,NaN,NaN,347082,NaN,G6,S
freq,NaN,NaN,NaN,1,577,NaN,NaN,NaN,7,NaN,4,644
mean,446.000000,0.383838,2.308642,NaN,NaN,29.699118,0.523008,0.381594,NaN,32.204208,NaN,NaN
std,257.353842,0.486592,0.836071,NaN,NaN,14.526497,1.102743,0.806057,NaN,49.693429,NaN,NaN
min,1.000000,0.000000,1.000000,NaN,NaN,0.420000,0.000000,0.000000,NaN,0.000000,NaN,NaN
25%,223.500000,0.000000,2.000000,NaN,NaN,20.125000,0.000000,0.000000,NaN,7.910400,NaN,NaN
50%,446.000000,0.000000,3.000000,NaN,NaN,28.000000,0.000000,0.000000,NaN,14.454200,NaN,NaN
75%,668.500000,1.000000,3.000000,NaN,NaN,38.000000,1.000000,0.000000,NaN,31.000000,NaN,NaN


## Feature Engineering

### Drop Irrelevant / High-Missing Columns

In [48]:
columns_to_drop = [
    "PassengerId",
    "Name",
    "Ticket",
    "Cabin",
    "Embarked"
]

df = df.drop(columns=columns_to_drop)
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare
0,0,3,male,22.0,1,0,7.2500
1,1,1,female,38.0,1,0,71.2833
2,1,3,female,26.0,0,0,7.9250
3,1,1,female,35.0,1,0,53.1000
4,0,3,male,35.0,0,0,8.0500


## Impute Age



In [49]:
df["Age"] = df["Age"].fillna(df["Age"].mean())

## Encode Sex

In [50]:
df["Sex"] = df["Sex"].map({"male": 0, "female": 1})

## Create FamilySize

In [51]:
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

## Log Transform Fare

In [52]:
df["Fare"] = np.log1p(df["Fare"])

## Feature / Target Split


In [53]:
TARGET = "Survived"

X = df.drop(columns=[TARGET])
y = df[TARGET]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (891, 7)
Target shape: (891,)


## Save Processed Dataset

In [54]:
# Create prefix-aware processed directory
processed_dir = f"data/processed/{DATA_PREP_PREFIX}"
os.makedirs(processed_dir, exist_ok=True)

processed_path = f"{processed_dir}/train_processed.csv"

df.to_csv(processed_path, index=False)
print("Saved processed dataset locally at:", processed_path)

# --- Push processed dataset to GCS (optional but portfolio-aligned) ---

from google.colab import auth
auth.authenticate_user()  # Ensure Colab has access to GCS

import subprocess
from pathlib import Path

# Local file that was just saved
local_processed_path = Path(processed_path)

# Verify the local file exists before uploading
if not local_processed_path.exists():
    raise FileNotFoundError(f"Local processed file not found: {local_processed_path}")
else:
    print(f"Saved processed dataset locally at: {local_processed_path}")

# Construct GCS URI (remove trailing slash if any)
GCS_URI = f"{DATA_PREP_PREFIX.rstrip('/')}/train_processed.csv"

# Upload local processed CSV to GCS
print(f"Uploading processed dataset to GCS at: {GCS_URI}")
subprocess.run(["gsutil", "cp", str(local_processed_path), GCS_URI], check=True)

# Confirm upload
!gsutil ls {DATA_PREP_PREFIX}
print("Upload complete.")

Saved processed dataset locally at: data/processed/gs://ml-cert-sandbox-bucket-js-001/data-prep//train_processed.csv
Saved processed dataset locally at: data/processed/gs:/ml-cert-sandbox-bucket-js-001/data-prep/train_processed.csv
Uploading processed dataset to GCS at: gs://ml-cert-sandbox-bucket-js-001/data-prep/train_processed.csv
gs://ml-cert-sandbox-bucket-js-001/data-prep/train_processed.csv
Upload complete.


## Feature Engineering Summary

- Dropped Cabin and Embarked
- Encoded Sex numerically
- Created FamilySize feature
- Imputed Age with mean
- Log-transformed Fare
- Saved processed dataset using DATA_PREP_PREFIX

#### Notes / Best Practices

1. **Colab GCS Authentication**  
   - `auth.authenticate_user()` ensures your Colab session has the credentials to read/write files in Google Cloud Storage.  
   - Always run this before any GCS operation to avoid 401 / Invalid Credentials errors.  
    - This allows pushing processed datasets to GCS without credential errors.

2. **Prefix-Driven Paths**  
   - `DATA_PREP_PREFIX` defines both the bucket and the folder structure dynamically.  
   - Avoids hardcoding bucket names or paths — this keeps notebooks portable and repeatable.  
   - Local and GCS paths are derived from this prefix.
   - Safe for iterative processing; overwrites previous runs if rerun.


3. **Safe Iterative Workflow**  
   - Local save (`df.to_csv(...)`) + GCS push ensures a deterministic workflow.  
   - Optionally uploaded to GCS at: `{DATA_PREP_PREFIX}/train_processed.csv`
   - Rerunning the notebook will **overwrite existing files** at the same path — safe for iterative processing.  

4. **Downstream Readiness**  
   - `03_training_keras_vertex.ipynb` can now reliably read the processed dataset from:  
     ```
     DATA_PREP_PREFIX/train_processed.csv
     ```  
   - No further manual copying is needed.  

5. **Portfolio Alignment**  
   - Pushing the processed dataset to GCS is **portfolio-grade**: demonstrates cloud readiness and reproducible preprocessing.  
   - Local + cloud copies together show good practice for versioning and data management.
   - Provides deterministic, reproducible output for model training.
   - Ensures alignment with Vertex AI workflows without launching jobs.
   - Clear separation between local artifacts and cloud storage.